# ML-07 — Baseline Action Score and Top-10 Review

**Lane:** Structured Content Archetype Clustering  
**Baseline action:** rank pages that look like **CTR opportunities for their current search position**  
**Development window:** March 2026 only  
**Future / label-derived inputs:** none

This notebook gives my Week-5 modelling work a simple rule baseline to beat. The rule is intentionally transparent: it uses only information available in the March decision window, checks the signals first, assigns one score, one reason code, and one action label, then reviews the top ten with a skeptical note.

I am keeping my clustering lane. The baseline is not the capstone model itself; it is a decision-support queue built from the same page-performance river.

### Safety
- No client names, URLs, titles, domains, or raw queries are displayed.
- June 2026 remains sealed.
- No FlyRank product flags (`needs_ctr_fix`, `is_quick_win`, `health_score`, etc.) are used as inputs.
- The CSV is generated locally under `work/outputs/` and should remain out of git.


## 0. Setup

The notebook reads the March 2026 warehouse partition directly from Hugging Face with a private `HF_TOKEN`.

In Colab:
1. Request access to `FlyRank/internship-warehouse`.
2. Add a plain **Read** token in the Secrets panel as `HF_TOKEN`.
3. Do not paste or print the token in a cell.
4. Run the notebook top to bottom.

The warehouse daily fact has one row per date, pseudonymized client, and pseudonymized content item. I aggregate it to one page-level row for March.


In [ ]:
%pip -q install duckdb pandas numpy

import os
import json
from pathlib import Path

import duckdb
import numpy as np
import pandas as pd
from IPython.display import display

# Private token: read from Colab Secrets first, environment second.
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get("HF_TOKEN")
except Exception:
    HF_TOKEN = os.getenv("HF_TOKEN")

if not HF_TOKEN:
    raise RuntimeError(
        "HF_TOKEN not found. Add it as a Colab Secret named HF_TOKEN, then rerun."
    )

con = duckdb.connect()
safe_token = HF_TOKEN.replace("'", "''")
con.execute(
    f"CREATE OR REPLACE SECRET hf_secret "
    f"(TYPE huggingface, TOKEN '{safe_token}')"
)

BASE = "hf://datasets/FlyRank/internship-warehouse"
MARCH = f"{BASE}/fact_content_daily_performance/month=2026-03/*.parquet"

# Show schema only so renamed warehouse fields fail early and visibly.
schema = con.sql(f"DESCRIBE SELECT * FROM read_parquet('{MARCH}')").df()
display(schema)

required = {
    "report_date",
    "client_id",
    "content_id",
    "gsc_impressions",
    "gsc_clicks",
    "gsc_avg_position",
}
missing = sorted(required - set(schema["column_name"]))
if missing:
    raise ValueError(f"Required warehouse columns are missing: {missing}")


## 1. Check two signals before writing the rule

My rule leans on two signals:

1. **CTR vs position** — this is directly tied to FlyRank's CTR-fix logic. If position is genuinely informative, pages ranking nearer the top should show higher weighted CTR than deeper-ranking pages.
2. **Impression volume** — I use volume only to prioritize the size of an opportunity. A low-CTR page with many impressions can matter more than the same gap on a page that is barely seen.

I will not assume either signal works. Each check prints a bucket table with **n** and ends with one verdict: `CONFIRMED`, `OPPOSITE`, `MIXED`, or `FALSE`.

### Page-level March frame

The aggregation below uses only March signals. `avg_position` is impression-weighted, and CTR is recomputed from total clicks / total impressions rather than averaging daily percentages.


In [ ]:
page_query = f"""
SELECT
    client_id,
    content_id,
    SUM(COALESCE(gsc_impressions, 0)) AS impressions,
    SUM(COALESCE(gsc_clicks, 0)) AS clicks,
    CASE
        WHEN SUM(COALESCE(gsc_impressions, 0)) > 0
        THEN 100.0 * SUM(COALESCE(gsc_clicks, 0))
             / SUM(COALESCE(gsc_impressions, 0))
    END AS ctr_pct,
    CASE
        WHEN SUM(CASE WHEN gsc_impressions > 0 THEN gsc_impressions ELSE 0 END) > 0
        THEN SUM(
            CASE
                WHEN gsc_impressions > 0 AND gsc_avg_position > 0
                THEN gsc_avg_position * gsc_impressions
                ELSE 0
            END
        ) / SUM(CASE WHEN gsc_impressions > 0 THEN gsc_impressions ELSE 0 END)
    END AS avg_position,
    COUNT(DISTINCT CASE WHEN gsc_impressions > 0 THEN report_date END) AS active_days
FROM read_parquet('{MARCH}')
GROUP BY 1, 2
HAVING SUM(COALESCE(gsc_impressions, 0)) > 0
"""

pages = con.sql(page_query).df()

# Local public-safe reference for notebook display only.
# The original pseudonymized IDs remain in the local CSV so later work can join back,
# but they are not printed in the notebook.
pages["page_ref"] = (
    pages["client_id"].astype(str) + "|" + pages["content_id"].astype(str)
).map(lambda x: __import__("hashlib").sha256(x.encode()).hexdigest()[:12])

print(f"March page-level rows: {len(pages):,}")
display(pages[["page_ref", "impressions", "clicks", "ctr_pct", "avg_position", "active_days"]].head())


### Signal check 1 — CTR vs position

**Expected direction:** better positions should have higher weighted CTR.

Buckets:
- `top_3`: average position ≤ 3
- `page_1_rest`: >3 and ≤10
- `page_2`: >10 and ≤20
- `deep`: >20

The table prints page count `n`, total impressions/clicks, and weighted CTR for every bucket.


In [ ]:
signal1 = pages.loc[
    pages["avg_position"].notna() & pages["ctr_pct"].notna()
].copy()

signal1["position_bucket"] = pd.cut(
    signal1["avg_position"],
    bins=[0, 3, 10, 20, np.inf],
    labels=["top_3", "page_1_rest", "page_2", "deep"],
    include_lowest=True,
)

position_buckets = (
    signal1.groupby("position_bucket", observed=False)
    .agg(
        n=("content_id", "size"),
        impressions=("impressions", "sum"),
        clicks=("clicks", "sum"),
        median_position=("avg_position", "median"),
    )
    .reset_index()
)

position_buckets["weighted_ctr_pct"] = (
    100.0 * position_buckets["clicks"]
    / position_buckets["impressions"].replace(0, np.nan)
)

display(position_buckets.round(3))

ctr_values = position_buckets["weighted_ctr_pct"].to_numpy(dtype=float)
valid = np.isfinite(ctr_values)

if valid.sum() < 3:
    signal1_verdict = "FALSE"
else:
    diffs = np.diff(ctr_values[valid])
    if np.all(diffs < 0):
        signal1_verdict = "CONFIRMED"
    elif np.all(diffs > 0):
        signal1_verdict = "OPPOSITE"
    elif np.nanmax(ctr_values) - np.nanmin(ctr_values) < 0.01:
        signal1_verdict = "FALSE"
    else:
        signal1_verdict = "MIXED"

print("Signal 1 verdict:", signal1_verdict)


### Signal check 2 — impression volume

**Expected direction:** higher-impression pages should carry more observed click volume, which supports using impressions as a prioritization weight rather than as a quality label.

I use quantile buckets so each bucket contains a meaningful number of pages. The check is deliberately modest: it does **not** claim that impressions cause quality or future success.


In [ ]:
signal2 = pages.loc[
    pages["impressions"].gt(0) & pages["clicks"].ge(0)
].copy()

# qcut can drop duplicate edges if the distribution is tied.
signal2["volume_bucket"] = pd.qcut(
    signal2["impressions"],
    q=4,
    labels=None,
    duplicates="drop",
)

volume_buckets = (
    signal2.groupby("volume_bucket", observed=False)
    .agg(
        n=("content_id", "size"),
        min_impressions=("impressions", "min"),
        median_impressions=("impressions", "median"),
        max_impressions=("impressions", "max"),
        median_clicks=("clicks", "median"),
        total_clicks=("clicks", "sum"),
    )
    .reset_index()
)

display(volume_buckets)

med_clicks = volume_buckets["median_clicks"].to_numpy(dtype=float)
if len(med_clicks) < 3:
    signal2_verdict = "FALSE"
else:
    diffs = np.diff(med_clicks)
    if np.all(diffs >= 0) and np.any(diffs > 0):
        signal2_verdict = "CONFIRMED"
    elif np.all(diffs <= 0) and np.any(diffs < 0):
        signal2_verdict = "OPPOSITE"
    elif np.nanmax(med_clicks) == np.nanmin(med_clicks):
        signal2_verdict = "FALSE"
    else:
        signal2_verdict = "MIXED"

print("Signal 2 verdict:", signal2_verdict)


### Signal-check interpretation

I will keep the baseline rule only if the **CTR-vs-position** check is not `FALSE` or `OPPOSITE`.

- `CONFIRMED`: the bucket pattern supports the direction I expected.
- `MIXED`: the signal is present but not clean enough to treat as a universal truth.
- `OPPOSITE`: the direction is reversed.
- `FALSE`: the bucket table does not support the signal.

A negative verdict is still useful because it tells me not to build a rule around a weak assumption.


In [ ]:
if signal1_verdict in {"FALSE", "OPPOSITE"}:
    print(
        "WARNING: CTR-vs-position was not supported. "
        "Do not treat the baseline queue below as a validated production rule."
    )
else:
    print(
        "CTR-vs-position is usable as a transparent baseline signal:",
        signal1_verdict
    )


## 2. Encode ONE rule and build the ranked queue

### Rule in plain words

> Among pages with enough March visibility and an average position between 1 and 20, rank pages higher when their CTR is further below the typical CTR for their own position bucket, with impression volume used as a smaller prioritization weight.

This is deliberately a **CTR opportunity review** rule, not a statement that a page is bad.

### Exactly one reason code

`LOW_CTR_FOR_POSITION`

### One action label

`REVIEW_CTR_OPPORTUNITY`

### Score

The score is 0–100:

- **70%** = relative CTR gap from the position-bucket benchmark
- **30%** = log-impression percentile inside the eligible candidate set

Eligibility guardrails:
- at least 100 March impressions;
- at least 7 active days;
- average position > 0 and ≤ 20;
- CTR below the benchmark for its position bucket.

No future month, product flag, health score, or label-derived feature enters the rule.


In [ ]:
MIN_IMPRESSIONS = 100
MIN_ACTIVE_DAYS = 7
MAX_POSITION = 20.0

rule = pages.copy()
rule = rule.loc[
    rule["impressions"].ge(MIN_IMPRESSIONS)
    & rule["active_days"].ge(MIN_ACTIVE_DAYS)
    & rule["avg_position"].gt(0)
    & rule["avg_position"].le(MAX_POSITION)
    & rule["ctr_pct"].notna()
].copy()

rule["position_bucket"] = pd.cut(
    rule["avg_position"],
    bins=[0, 3, 10, 20],
    labels=["top_3", "page_1_rest", "page_2"],
    include_lowest=True,
)

# Benchmark = weighted CTR of the eligible pages in each position bucket.
benchmarks = (
    rule.groupby("position_bucket", observed=False)
    .agg(bucket_clicks=("clicks", "sum"), bucket_impressions=("impressions", "sum"))
)
benchmarks["expected_ctr_pct"] = (
    100.0 * benchmarks["bucket_clicks"]
    / benchmarks["bucket_impressions"].replace(0, np.nan)
)

rule = rule.join(
    benchmarks["expected_ctr_pct"],
    on="position_bucket",
)

rule["ctr_gap_pct_points"] = rule["expected_ctr_pct"] - rule["ctr_pct"]

# Only real underperformance versus its own position benchmark enters the queue.
queue = rule.loc[rule["ctr_gap_pct_points"].gt(0)].copy()

# Relative gap, capped to [0, 1].
queue["relative_ctr_gap"] = (
    queue["ctr_gap_pct_points"]
    / queue["expected_ctr_pct"].replace(0, np.nan)
).clip(lower=0, upper=1)

# Volume is a prioritization weight, not a label.
queue["log_impressions"] = np.log1p(queue["impressions"])
queue["volume_percentile"] = queue["log_impressions"].rank(pct=True)

queue["baseline_action_score"] = (
    100.0 * (
        0.70 * queue["relative_ctr_gap"].fillna(0)
        + 0.30 * queue["volume_percentile"].fillna(0)
    )
).round(2)

queue["reason_code"] = "LOW_CTR_FOR_POSITION"
queue["action_label"] = "REVIEW_CTR_OPPORTUNITY"

queue = queue.sort_values(
    ["baseline_action_score", "impressions"],
    ascending=[False, False],
).reset_index(drop=True)

queue["rank"] = np.arange(1, len(queue) + 1)

display_cols = [
    "rank",
    "page_ref",
    "baseline_action_score",
    "action_label",
    "reason_code",
    "impressions",
    "ctr_pct",
    "avg_position",
    "expected_ctr_pct",
    "ctr_gap_pct_points",
    "active_days",
]

print(f"Eligible ranked pages: {len(queue):,}")
display(queue[display_cols].head(10).round(3))


### Write the queue

The CSV is regenerated on every run and remains out of git by design.

I keep the warehouse pseudonymized IDs in the local CSV so later notebooks can join the queue back to the warehouse. The notebook display uses only `page_ref`.


In [ ]:
OUTPUT_DIR = Path("work/outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

csv_path = OUTPUT_DIR / "baseline_action_score.csv"

queue_export_cols = [
    "rank",
    "client_id",
    "content_id",
    "baseline_action_score",
    "reason_code",
    "action_label",
    "impressions",
    "clicks",
    "ctr_pct",
    "avg_position",
    "expected_ctr_pct",
    "ctr_gap_pct_points",
    "active_days",
]

queue[queue_export_cols].to_csv(csv_path, index=False)

print(f"Wrote {len(queue):,} ranked rows to: {csv_path}")


## 3. Top-10 review

For each of the top ten, I want one skeptical line that answers:

- **Action:** what I would do;
- **Why:** why this page is in the queue;
- **What would make it wrong:** a plausible reason the rule could be misleading.

The review does not claim that the page definitely needs a change. CTR can be affected by query intent, SERP features, branded/non-branded mix, title/snippet fit, or unstable data that this page-level rule cannot see.


In [ ]:
top10 = queue.head(10).copy()

def review_line(row):
    wrong = (
        "wrong if query intent/brand mix or SERP features explain the lower CTR, "
        "or if March is not representative"
    )
    return (
        f"{row['action_label']} — "
        f"score {row['baseline_action_score']:.1f}; "
        f"{int(row['impressions']):,} impressions; "
        f"CTR {row['ctr_pct']:.2f}% vs "
        f"{row['expected_ctr_pct']:.2f}% for its position bucket; "
        f"{wrong}."
    )

top10["review"] = top10.apply(review_line, axis=1)

top10_review = top10[
    [
        "rank",
        "page_ref",
        "baseline_action_score",
        "action_label",
        "reason_code",
        "review",
    ]
].copy()

pd.set_option("display.max_colwidth", 220)
display(top10_review)

print("\nOne-line skeptical review:")
for _, row in top10_review.iterrows():
    print(f"{int(row['rank'])}. {row['page_ref']}: {row['review']}")


## 4. Weak picks + leakage check

### What I expect to be weak

Even a transparent baseline can rank the wrong pages highly. The most likely weak picks are:

- pages barely above the impression or active-day guardrail;
- pages whose CTR gap is driven by a query mix the page-level data hides;
- pages affected by SERP features that naturally suppress clicks;
- pages whose March behavior is unusual rather than representative.

I inspect the top ten for those warning signs below.

### Leakage policy

The rule may use only March measurements available at the decision moment. It must not use:
- April/May/June outcomes;
- `needs_ctr_fix`, `is_quick_win`, `health_score`, or other FlyRank product flags;
- any future label or direct copy of a label;
- titles, URLs, domains, or raw queries.


In [ ]:
weak_picks = top10.loc[
    (top10["impressions"] < 2 * MIN_IMPRESSIONS)
    | (top10["active_days"] < 2 * MIN_ACTIVE_DAYS)
].copy()

if weak_picks.empty:
    print(
        "No top-10 row is obviously weak from the two simple stability guardrails. "
        "That does not prove the picks are correct; query-level and SERP context can still overturn them."
    )
else:
    weak_picks["weak_pick_note"] = np.where(
        weak_picks["impressions"] < 2 * MIN_IMPRESSIONS,
        "Near the minimum impression threshold; CTR may be less stable.",
        "Observed on relatively few active days; March may be less representative.",
    )
    display(
        weak_picks[
            ["rank", "page_ref", "impressions", "active_days", "weak_pick_note"]
        ]
    )

# Explicit leakage audit over the columns used by the score.
score_input_columns = {
    "impressions",
    "ctr_pct",
    "avg_position",
    "active_days",
    "expected_ctr_pct",
    "ctr_gap_pct_points",
    "relative_ctr_gap",
    "log_impressions",
    "volume_percentile",
}

forbidden_fragments = {
    "future",
    "label",
    "target",
    "health_score",
    "needs_ctr_fix",
    "is_quick_win",
    "recommended_action",
    "action_type",
    "june",
    "april",
    "may",
}

hits = sorted(
    col
    for col in score_input_columns
    if any(fragment in col.lower() for fragment in forbidden_fragments)
)

assert not hits, f"Leakage-risk score inputs found: {hits}"
print("PASS: no future-window, label-derived, or FlyRank product-flag input is used by the score.")


## 5. Save run receipts

The CSV stays out of git, but this small JSON is safe to commit as the run receipt. It contains aggregate metadata only.


In [ ]:
metrics = {
    "lane": "structured_content_archetype_clustering",
    "development_month": "2026-03",
    "signal_1": {
        "name": "ctr_vs_position",
        "verdict": signal1_verdict,
        "flag_linked": True,
    },
    "signal_2": {
        "name": "impression_volume",
        "verdict": signal2_verdict,
        "flag_linked": False,
    },
    "rule": {
        "reason_code": "LOW_CTR_FOR_POSITION",
        "action_label": "REVIEW_CTR_OPPORTUNITY",
        "min_impressions": MIN_IMPRESSIONS,
        "min_active_days": MIN_ACTIVE_DAYS,
        "max_position": MAX_POSITION,
        "score_weights": {
            "relative_ctr_gap": 0.70,
            "volume_percentile": 0.30,
        },
    },
    "page_rows_in_march": int(len(pages)),
    "ranked_queue_rows": int(len(queue)),
    "top10_reviewed": int(min(10, len(queue))),
    "leakage_check_passed": len(hits) == 0,
}

json_path = OUTPUT_DIR / "baseline_action_score_metrics.json"
json_path.write_text(json.dumps(metrics, indent=2), encoding="utf-8")

print(json.dumps(metrics, indent=2))
print(f"\nWrote metrics receipt to: {json_path}")


## Self-check

Before submitting, confirm each line honestly:

- [x] Lane is confirmed: Structured Content Archetype Clustering.
- [x] Two signals are checked before the rule.
- [x] At least one signal is tied to a real FlyRank flag: CTR vs position.
- [x] Each signal check prints a bucket table with `n`.
- [x] Each signal gets one verdict: CONFIRMED / OPPOSITE / MIXED / FALSE.
- [x] Exactly one baseline rule is encoded.
- [x] The rule produces one score, one reason code, and one action label.
- [x] The notebook writes `work/outputs/baseline_action_score.csv`.
- [x] The top ten each receive a skeptical one-line review.
- [x] Weak picks are checked.
- [x] No future-window, label-derived, or FlyRank product-flag input is used in the score.
- [x] A metrics JSON is written for commit.
- [ ] Run all cells with warehouse access and make sure the real tables/outputs are visible.
- [ ] Save this notebook as `work/notebooks/w04_baseline_score.ipynb`.
- [ ] Commit the notebook and `work/outputs/baseline_action_score_metrics.json`.
- [ ] Do **not** commit `baseline_action_score.csv`.
- [ ] Submit the repo URL on the assignment card.
